# КР по модулю 1 — практическая часть



## Задание 1. (1 балл)

Используйте корпус `lenta.txt` из семинара 2 (ячейка ниже скачает его).

In [1]:
import os
import urllib.request
import zipfile

os.makedirs("data", exist_ok=True)

ZIP_URL = "https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip"
ZIP_PATH = "data/lenta.txt.zip"
TXT_PATH = "data/lenta.txt"

if not os.path.exists(TXT_PATH):
    print("Скачиваю", ZIP_URL)
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("data")
    print("Готово.")

text = open(TXT_PATH, encoding="utf-8").read()
print("Длина текста:", len(text))
print("Первые 300 символов:", text[:300])

Длина текста: 11536552
Первые 300 символов: Бои у Сопоцкина и Друскеник закончились отступлением германцев. Неприятель, приблизившись с севера к Осовцу начал артиллерийскую борьбу с крепостью. В артиллерийском бою принимают участие тяжелые калибры. С раннего утра 14 сентября огонь достиг значительного напряжения. Попытка германской пехоты про


In [ ]:
import re
from collections import Counter

# Слово: последовательность букв (кириллица/латиница/цифры/дефис), в которой есть хотя бы одна кириллическая и хотя бы одна латинская буква. Границы слова - (?<!\w) и (?!\w), чтобы не резать составные слова
# Внутри разрешаем кириллицу, латиницу, цифры, дефис и апостроф
MIXED_PATTERN = re.compile(
    r"(?<!\w)(?=[^\s]*[А-Яа-яЁё])(?=[^\s]*[A-Za-z])[А-Яа-яЁёA-Za-z0-9\-']+(?!\w)",
    flags=re.UNICODE,
)

matches = MIXED_PATTERN.findall(text)
print("Всего совпадений:", len(matches))
print("Первые 20:", matches[:20])

counter = Counter(matches)
top = counter.most_common(10)
print("\nТоп-10 смешанных слов:")
for w, c in top:
    print(f"  {w}: {c}")

if top:
    print("\nСамое частое:", top[0][0])
else:
    print("\nСмешанных слов не найдено.")

Всего совпадений: 1120
Первые 20: ['машинs', 'Mhz', 'Coopers', 'Scientist', 'http', 'передачи', 'Cледующим', 'отправится', 'на', 'России', 'cсылкой', 'Cпецназ', 'корреспонденту', 'делу', 'управлением', 'в', 'подаcт', 'газету', 'web-сайта', 'web-проекта']

Топ-10 смешанных слов:
  Яndex: 38
  Reuters: 13
  com: 13
  BBC: 11
  Press: 10
  web-сайтов: 8
  web-страниц: 7
  cообщает: 7
  web-страницы: 7
  Поколение: 7

Самое частое: Яndex


Одним регулярным выражением найдите в **исходном тексте** корпуса (без какой-либо предобработки) примеры слов, в которых совмещается кириллица и латиница. Найдите самое частотное такое слово и вставьте его в форму как ответ.

### Задание 2 (1 балл)

Дана последовательность слов:

`об этом сообщает агентство интерфакс со ссылкой на пресс-службу`

Приведите весь текст корпуса `data/lenta.txt` к нижнему регистру и сделайте простую токенизацию: разбейте все по пробелам, удалите пунктуацию по краям токенов (`strip`) и отфильтруйте пустые строки, оставшиеся после этих шагов.

По частотам из корпуса рассчитайте для каждого триграмма в последовательности условную вероятность P(w3 | w1 w2) = count(w1 w2 w3) / count(w1 w2). Вставьте в форму триграмм с самой низкой вероятностью (три слова через пробел).

*если где-то получается деление но ноль, то пропустите это сочетание

In [3]:
import re
from collections import Counter

low = text.lower()

tokens = low.split()

def strip_punct(tok: str) -> str:
    return tok.strip('.,!?;:"\'«»()[]{}—–-…')

tokens = [strip_punct(t) for t in tokens]
tokens = [t for t in tokens if t]

print("Всего токенов:", len(tokens))

bigrams  = Counter(zip(tokens, tokens[1:]))
trigrams = Counter(zip(tokens, tokens[1:], tokens[2:]))

seq = "об этом сообщает агентство интерфакс со ссылкой на пресс-службу".split()

print("\nТриграммы и их условные вероятности:")
results = []
for i in range(len(seq) - 2):
    w1, w2, w3 = seq[i], seq[i + 1], seq[i + 2]
    bg = (w1, w2)
    tg = (w1, w2, w3)

    c_bg = bigrams.get(bg, 0)
    c_tg = trigrams.get(tg, 0)

    if c_bg == 0:        
        print(f"  [{w1} {w2} {w3}]  P = —  (count({w1} {w2}) = 0, пропущено)")
        continue

    p = c_tg / c_bg
    print(f"  [{w1} {w2} {w3}]  P = {c_tg}/{c_bg} = {p:.6f}")
    results.append((w1, w2, w3, p))

if results:
    worst = min(results, key=lambda r: r[3])
    answer = " ".join(worst[:3])
    print("\nСамая низкая вероятность:", answer)
    print(f"  P = {worst[3]:.6f}")
else:
    print("\nВсе триграммы пропущены (деление на ноль).")

Всего токенов: 1492507

Триграммы и их условные вероятности:
  [об этом сообщает]  P = 284/1776 = 0.159910
  [этом сообщает агентство]  P = 22/288 = 0.076389
  [сообщает агентство интерфакс]  P = 7/344 = 0.020349
  [агентство интерфакс со]  P = 2/27 = 0.074074
  [интерфакс со ссылкой]  P = 180/186 = 0.967742
  [со ссылкой на]  P = 1241/1251 = 0.992006
  [ссылкой на пресс-службу]  P = 139/1247 = 0.111468

Самая низкая вероятность: сообщает агентство интерфакс
  P = 0.020349


### Задание 3 (1 балл)

Загрузите датасет новостей `lenta_40k` из семинара 3 (ячейка ниже). Сделайте подвыборку из 500 текстов: `lenta.sample(500, random_state=8)`.

Посчитайте эмбеддинги этих текстов моделью `cointegrated/rubert-tiny2` (без префиксов). Рассчитайте косинусную близость всех со всеми (для каждой пары текстов) и найдите самую близкую пару текстов. 

Вставьте в форму первые слова обоих текстов через пробел (например, `Холдинг Американская`, порядок не важен).

!NB: Тексты должны быть разными

In [4]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

lenta = pd.read_csv(
    "https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta_40k.csv.zip"
).sample(500, random_state=8)

print("Размер выборки:", lenta.shape)
print("Колонки:", lenta.columns.tolist())

MODEL_NAME = "cointegrated/rubert-tiny2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).eval().to("cpu")
torch.set_num_threads(4)


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


@torch.no_grad()
def embed(texts, batch_size=32):
    """rubert-tiny2: без префиксов, mean pooling, L2-нормализация"""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(model.device)
        out = model(**enc)
        emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        all_embs.append(emb.cpu().numpy())
    return np.vstack(all_embs)


doc_texts = lenta["text"].astype(str).tolist()
doc_embs = embed(doc_texts)
print("Эмбеддинги документов:", doc_embs.shape)

sims = doc_embs @ doc_embs.T

np.fill_diagonal(sims, -1.0)

best_i, best_j = np.unravel_index(np.argmax(sims), sims.shape)
best_score = float(sims[best_i, best_j])

first_words_1 = doc_texts[best_i].split()[0] if doc_texts[best_i].split() else ""
first_words_2 = doc_texts[best_j].split()[0] if doc_texts[best_j].split() else ""

print(f"\nЛучшая пара: ({best_i}, {best_j}), score = {best_score:.4f}")
print(f"\nТекст 1 (первые 300): {doc_texts[best_i][:300]!r}")
print(f"Текст 2 (первые 300): {doc_texts[best_j][:300]!r}")
print(f"\nПервые слова: {first_words_1!r} и {first_words_2!r}")

triu = np.triu_indices(len(doc_texts), k=1)
pairs_scores = sims[triu]
top5 = np.argsort(-pairs_scores)[:5]

print("\nТоп-5 близких пар:")
for rank, k in enumerate(top5, 1):
    i, j = triu[0][k], triu[1][k]
    s = pairs_scores[k]
    print(f"  {rank}. score={s:.4f} | ({i},{j}) "
          f"| {doc_texts[i][:60]!r}  ×  {doc_texts[j][:60]!r}")

c:\Users\1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Размер выборки: (500, 2)
Колонки: ['text', 'topic']


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 27571.02it/s]


Эмбеддинги документов: (500, 312)

Лучшая пара: (301, 391), score = 0.9657

Текст 1 (первые 300): 'Нападающий «Бастии» Брандао появился на заседании Федерации футбола Франции вместе со своей матерью. Там обсуждалась его апелляция на решение дисциплинарного комитета французской Профессиональной футбольной лиги о дисквалификации Брандао на шесть месяцев за то, что он ударил игрока ПСЖ Тиаго Мотту. '
Текст 2 (первые 300): "Футболист французского клуба «Бастия» Брандао, сломавший нос игроку ПСЖ Тьягу Мотте, решением дисциплинарного комитета французской Профессиональной футбольной лиги дисквалифицирован на полгода. Об этом сообщает французское издание L'Equpe. Срок дисквалификации игрока де-юре начался 22 августа и исте"

Первые слова: 'Нападающий' и 'Футболист'

Топ-5 близких пар:
  1. score=0.9657 | (301,391) | 'Нападающий «Бастии» Брандао появился на заседании Федерации '  ×  'Футболист французского клуба «Бастия» Брандао, сломавший нос'
  2. score=0.9563 | (205,310) | 'Утром 18 июля кур